### SFOS API - IPv4 Firewall Rules Management

**Purpose:** Create, read, update, and delete IPv4 firewall rules

**Topics:**
- Basic CRUD operations using domain StrEnum objects
- Firewall rule creation with various configurations
- Pagination and filtering
- Error handling
- Bulk operations
- Rule positioning and ordering

**Requirements:**
- Environment variables configured (.env file)
- SFOS firewall accessible
- API key with appropriate permissions

---

#### Setup: Import Libraries and Configure Client

In [1]:
from common import print_data, setup_client

from sfos_api import Firewall

# Initialize the SFOS API Client
client = setup_client()
print("✅ SFOS API client initialized")

✅ SFOS API client initialized


#### Create IPv4 Firewall Rules

Create firewall rule objects using the `Firewall.RULES_IPV4` endpoint object.

Firewall rules define how traffic is allowed, denied, or processed by the firewall.

API Link: [Create IPv4 rule | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/create-firewall-rule-ipv-4/)

In [5]:
# Create an allow rule
# for allowing HTTP traffic from the internal network
# to the external network
#
endpoint = Firewall.RULES_IPV4
rule_name = "test-rule-allow-http"

payload = {
    "name": rule_name,
    "description": "Allow HTTP traffic from internal network",
    "ruleType": "firewall",
    "enabled": True,
    "action": "accept",
    "position": "top",
    "sourceZones": {"zones": [{"name": "LAN"}]},
    "sourceNetworks": {
        "ipv4Addresses": [
            {"name": "test-ipv4-group-internal"},
        ],
    },
    "destinationZones": {"zones": [{"name": "WAN"}]},
    "destinationNetworks": {"any": True},
    "servicesOrGroups": {
        "services": [
            {"name": "HTTP"},
            {"name": "HTTPS"},
        ]
    },
    "logTraffic": True,
}

try:
    response = client.create(endpoint, payload)
    print(f"✅ Created firewall rule '{rule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating firewall rule: {e}")
    print_data(str(e))

✅ Created firewall rule 'test-rule-allow-http'
{
  "action": "accept",
  "description": "Allow HTTP traffic from internal network",
  "destinationNetworks": {
    "any": true
  },
  "destinationZones": {
    "zones": [
      {
        "name": "WAN"
      }
    ]
  },
  "emailScanning": {
    "imap": false,
    "imaps": false,
    "pop3": false,
    "pop3s": false,
    "smtp": false,
    "smtps": false
  },
  "enabled": true,
  "exclusions": {
    "destinationNetworks": {
      "countries": [],
      "countryGroups": [],
      "fqdnAddresses": [],
      "fqdnGroups": [],
      "ipv4Addresses": [],
      "ipv4Groups": [],
      "macAddresses": []
    },
    "destinationZones": {
      "zones": []
    },
    "servicesOrGroups": {
      "serviceGroups": [],
      "services": []
    },
    "sourceNetworks": {
      "countries": [],
      "countryGroups": [],
      "fqdnAddresses": [],
      "fqdnGroups": [],
      "ipv4Addresses": [],
      "ipv4Groups": [],
      "macAddresses": []
    },


In [6]:
# Create a deny rule
endpoint = Firewall.RULES_IPV4
rule_name = "test-rule-deny-telnet"

payload = {
    "name": rule_name,
    "description": "Deny Telnet traffic",
    "action": "deny",
    "sourceAddresses": [
        {"name": "Any"},
    ],
    "destinationAddresses": [
        {"name": "Any"},
    ],
    "services": [
        {"name": "Telnet"},
    ],
    "logTraffic": True,
}

try:
    response = client.create(endpoint, payload)
    print(f"✅ Created firewall rule '{rule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating firewall rule: {e}")
    print_data(str(e))

Error creating firewall rule: Invalid action.
Invalid action.


#### Read IPv4 Firewall Rules

Retrieve firewall rule information by ID/name or list all rules.

API Links: [Get all IPv4 rules | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/get-all-firewall-rules-ipv-4/) | [Get IPv4 rule | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/get-firewall-rule-ipv-4/)

In [ ]:
# Get a specific firewall rule by name
endpoint = Firewall.RULES_IPV4
rule_name = "test-rule-allow-http"

try:
    response = client.read(endpoint, id_or_name=rule_name)
    print(f"✅ Retrieved firewall rule '{rule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error reading firewall rule: {e}")
    print_data(str(e))

In [ ]:
# Get all firewall rules
endpoint = Firewall.RULES_IPV4

try:
    response = client.read(endpoint)

    if "items" in response and response["items"]:
        print(f"✅ Retrieved {len(response['items'])} firewall rules")
        # Show first 3 rules
        print_data(response["items"][:3])
        if len(response["items"]) > 3:
            print(f"... (showing first 3 of {len(response['items'])} rules)")
    else:
        print("No firewall rules found")
except Exception as e:
    print(f"Error reading firewall rules: {e}")
    print_data(str(e))

In [ ]:
# Get firewall rules with filtering
endpoint = Firewall.RULES_IPV4
params = {
    "nameContains": "test-rule-",
    "page": 1,
    "pageSize": 10,
}

try:
    response = client.read(endpoint, params=params, pageTotal=False)

    if "items" in response:
        print(f"✅ Retrieved {len(response['items'])} firewall rules containing 'test-rule-'")
        print_data(response["items"])
    else:
        print("No firewall rules found matching filter")
except Exception as e:
    print(f"Error reading firewall rules: {e}")
    print_data(str(e))

#### Update IPv4 Firewall Rule

Modify an existing firewall rule's properties.

API Link: [Update IPv4 rule | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/update-firewall-rule-ipv-4/)

In [ ]:
# Update a firewall rule
endpoint = Firewall.RULES_IPV4
rule_name = "test-rule-allow-http"

payload = {
    "description": "Allow HTTP and HTTPS traffic from internal network to external",
    "services": [
        {"name": "HTTP"},
        {"name": "HTTPS"},
        {"name": "DNS"},
    ],
}

try:
    response = client.update(endpoint, id_or_name=rule_name, data=payload)
    print(f"✅ Updated firewall rule '{rule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error updating firewall rule: {e}")
    print_data(str(e))

#### Move IPv4 Firewall Rule

Reposition a firewall rule within the ordered rulebase using `client.move()`.

API Link: [Move IPv4 rule | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/move-firewall-rule-ipv-4/)


In [ ]:
# Move a firewall rule within the rulebase
endpoint = Firewall.RULES_IPV4
rule_name = "test-rule-allow-http"

move_data = {
    "id": rule_name,
    "position": "top",
}

try:
    response = client.move(endpoint, move_data)
    print(f"✅ Moved firewall rule '{rule_name}' to position 'top'")
    print_data(response)
except Exception as e:
    print(f"Error moving firewall rule: {e}")
    print_data(str(e))


#### Delete IPv4 Firewall Rules

Remove firewall rule objects (single or bulk delete).

API Links: [Delete IPv4 rule | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/delete-firewall-rule-ipv-4/) | [Bulk delete IPv4 rules | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/bulk-delete-firewall-rules-ipv-4/)

In [ ]:
# Delete a single firewall rule
endpoint = Firewall.RULES_IPV4
rule_name = "test-rule-allow-http"

try:
    response = client.delete(endpoint, id_or_name=rule_name)
    print(f"✅ Deleted firewall rule '{rule_name}'")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error deleting firewall rule: {e}")
    print_data(str(e))

In [ ]:
# Bulk delete firewall rules
endpoint = Firewall.RULES_IPV4

items_to_delete = [
    {"name": "test-rule-allow-http"},
    {"name": "test-rule-deny-telnet"},
]

try:
    response = client.delete(endpoint, items=items_to_delete)
    print(f"✅ Bulk deleted {len(items_to_delete)} firewall rules")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error bulk deleting firewall rules: {e}")
    print_data(str(e))

In [ ]:
# Close the client session when done
client.close()
print("✅ Client session closed")